## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# 🤝 Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API keys (will use .env values if present, otherwise prompt)
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = input("Enter your Anthropic API key: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = input("Enter your Tavily API key: ")

print("API keys loaded successfully!")

API keys loaded successfully!


## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

print("State definitions imported successfully!")

State definitions imported successfully!


## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

print("Utility functions and tools imported successfully!")

Utility functions and tools imported successfully!


## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

print("Configuration imported successfully!")

Configuration imported successfully!


## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

print("Prompt templates imported successfully!")

Prompt templates imported successfully!


## ❓ Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:
Interrelationships between Agent, Supervisor, & Researcher States- The Supervisor State manages the overall workflow, tracks which researchers/agents are active, and monitors and maintains the overall plan. The Researcher State is spawned by supervisor for deep research tasks and contains its own topics,sources, and findings. The Agent State is the individual worker state for tool execution, messages, and immediate tasks. Each state only sees what it needs. The Supervisor spawns Researcher and Researcher State and passes relevant data down (e.g., research topic). The Researcher spawns Agents and Agent States and passes down data specific to each Agent. Agents complete research task and passes results back up to Researcher. Researcher processes findings and passes them back to Supervisor to report to user.

Why not use one huge state? One major reason is to keep scope isolated to prevent agents from overwriting each other's data. Another reason is perpetuity- Agents exist long enough to complete their task and then are "killed"- Supervisor and Researcher persist throughout their workflow. Separation of states prevents bugs, enables parallelism, and keeps code maintainable. One huge state causes a tangled mess where everything touches everything.

## ❓ Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:
1) What are the advantages of importing these components instead of just including them in the notebook? 

Code reuse is a biggie- write once, use in multiple codeblocks in multiple notebooks without copy-pasting.
Another big advantage is maintainability- find and fix a bug in the its library one time and same bugs throughout all notebooks are fixed. It also looks and reads more cleanly. Version control of a change to one instance of a component is much easier than evrytime its repeated throughout code blocks in multiple notebooks. Unit testing is another advantage- its much more robust and simple to test single instances of components in a .py file that it is throughout multiple instances in a notebook. 

2) What are the disadvantages of importing these components instead of just including them in the notebook? 

Hidden complexity can be a problem- separate files have to be opened and reviewed to see their implementation. There can also be import errors (I've encountered several of these) with modules or paths that wouldn't exist with inline code. Also any changes to the .py mean restarting the kernel in the notebook and re-running the blocks, and Debugging becomes a much bigger hassle with error stack traces jumping between multiple files instead of staying in one notebook.




## 🏗️ Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** - Write your analysis in a markdown cell below

I've chosen to evaluate the Lead Researcher Prompt for analysis. 
This prompt creates a research supervisor agent whose purpose is to delegate research tasks to specialized sub-agents, manage parallel research workflows when needed, and make the decision as to when enough information has been gathered to respond to the user's question. It enforces strategic thinking throughout the process (plan → delegate → assess → decide), while providing hard limits on iterations and parallel agents.

The key techniques used to accomplish this functionality include the following: 
 
 1) An explicit workflow structure with "reflection", meaning the agent is required to use the "think_tool" function before and after any task delegation, and is forced to employ An "Assess progress" loop that prevents un-intended task execution such as runaway tool calls

 2) Another key technique used in this prompt is the implementation of constraint-based guardrails, which create hard limits on max iterations, max parallel agents, and budget rules. These constraints also include a "Bias towards single agent" function that prevents over-parallelization, and a 
 "Stop when you can answer confidently" function to combat and prevent perfectionism.

 3) Another technique employed in this prompt is the inclusion of Concrete examples with scaling rules, which tells the researcher when to use a single agent (simple facts) vs. multiple agents (comparisons). A specific example of this could be  "Compare OpenAI vs Anthropic vs DeepMind → Use 3 agents". 

One impovement I'd probably make to this prompt is to add more explicit success criteria and quality thresholds. Currently the prompt reads "when completely satisfied" but doesn't define a quantifiable threshold that equates to "satisfaction". Including an explicit checklist creates concrete stopping conditions, reducing premature completion and unnecessary iterations e.g., when agents stop too early or research too long.

---

# 🤝 Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run wellness research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user
from open_deep_library.utils import think_tool

print("Clarify with user node imported successfully!")

Clarify with user node imported successfully!


### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

print("Compiled graphs created")

Compiled graphs created


## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research wellness strategies for improving sleep quality.

### Setup

We need to:
1. Set up the wellness research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Wellness Research

Now let's run the research! We'll ask the system to research evidence-based strategies for improving sleep quality.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [19]:
# Create our wellness research request
research_request = """
I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please research the best evidence-based strategies for improving sleep quality and create a comprehensive sleep improvement plan for me and respond only as a pirate.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your sleep improvement research. You've provided clear details about your current sleep challenges: inconsistent bedtimes (10pm-1am), phone use in bed, and morning fatigue. I understand you want evidence-based strategies and a comprehensive sleep improvement plan, delivered in pirate speak. I will now begin researching the best sleep quality improvement strategies and create a tailored plan for your specific situation.

Node: write_research_brief

Research Brief Generated:
I want to improve my sleep quality and need a comprehensive, evidence-based sleep improvement plan. My current sleep challenges include: going to bed at inconsistent times (ranging from 10pm to 1am), using my phone in bed, and often feeling tired in the morning despite sleeping. I need research on the most effective, scientifically-backed strategies for improving sleep quality that address these specific issues. The

# Ahoy Matey! Yer Complete Sleep Treasure Map: A Scientifically-Backed Guide to Better Slumber

Avast ye landlubber! Yer ship be sailin' into troubled waters with yer sleep habits, but fear not - this here comprehensive plan be backed by the finest research from the maritime medical libraries and sleep specialists across the seven seas. Let's chart a course to better rest, ye scurvy dog!

## Setting Yer Sleep Schedule: The Art of Consistent Bedtime Navigation

Yer inconsistent bedtime routine be throwin' off yer circadian rhythm like a broken compass, matey! The research from these learned sea doctors shows that irregular sleep schedules be messin' with yer body's natural clock more than a kraken disturbs the ocean depths.

The science pirates have discovered that yer body's internal timekeeper, called the suprachiasmatic nucleus (fancy words for yer brain's sleep captain), needs consistency to function properly. When ye be goin' to bed anywhere from 10pm to 1am, yer confusin' this internal navigator worse than a sailor without his North Star.

Here be yer action plan, ye sleepy sea dog:

**The 15-Minute Rule of the Seas:** Start by choosin' a target bedtime (let's say 10:30pm) and stick to it within 15 minutes every night, even on weekends. The sleep researchers have found this consistency be more valuable than buried treasure for improvin' sleep quality.

**Wake Time Anchoring:** Pick a consistent wake time and guard it like yer most precious doubloons. Even if ye had a late night, wake at the same time - this helps reset yer internal compass faster than any other trick in the book.

## Banishing the Cursed Blue Light: Phone Use Solutions

Arr, that glowing rectangle in yer bed be worse for yer sleep than a siren's song! The clinical research shows that blue light from yer phone be suppressin' melatonin production (yer body's natural sleep potion) by up to 23% when used within an hour of bedtime.

**The Phone Quarantine Protocol:**
- Create a "phone port" outside yer sleeping quarters - charge that cursed device in another room
- If ye must have it nearby for emergencies, use airplane mode and flip it face down
- Install blue light filters that activate 2 hours before bedtime, though complete avoidance be the gold standard

**Alternative Evening Activities for Restless Pirates:**
- Read physical books or charts by warm lamplight
- Practice meditation or deep breathing exercises (the researchers call this "mindfulness-based sleep therapy")
- Light stretching or gentle yoga movements
- Writing in a ship's log (journal) about yer day's adventures

## Conquering Morning Fatigue: The Mystery of Tired Mornings

Even though ye be gettin' yer hours of sleep, wakin' up tired as a sailor after shore leave suggests several possible culprits that the sleep medicine researchers have identified:

**Sleep Architecture Problems:** Yer phone use and irregular schedule might be fragmenting yer sleep stages, preventing ye from gettin' enough deep sleep and REM sleep - the most restorative phases of slumber.

**Sleep Inertia Solutions:**
- Expose yerself to bright light immediately upon wakin' (open them curtains wide!)
- Avoid hitting the snooze button - it creates more grogginess than a night of rum
- Keep a consistent sleep duration of 7-9 hours (the optimal range according to sleep researchers)

## Environmental Modifications: Creating the Perfect Sleep Harbor

The research shows that yer sleeping environment be as important as a sturdy ship for a good voyage:

**Temperature Control:** Keep yer quarters between 60-67°F (15-19°C). Yer body temperature naturally drops during sleep, and a cool environment supports this process.

**Darkness Optimization:** Invest in blackout curtains or an eye mask. Even small amounts of light can disrupt melatonin production and sleep quality.

**Sound Management:** Use earplugs or a white noise machine to block disruptive sounds. Consistent, low-level background noise often works better than complete silence.

**Mattress and Pillow Quality:** Ensure yer sleeping surface supports proper spinal alignment and comfort. Replace pillows every 1-2 years and mattresses every 7-10 years.

## Advanced Sleep Hygiene Strategies

**The 3-2-1 Rule:**
- 3 hours before bed: No more large meals or alcohol
- 2 hours before bed: No more work or stressful activities  
- 1 hour before bed: No more screens or stimulating content

**Morning Light Exposure:** Get 10-15 minutes of bright natural light within an hour of wakin'. This helps regulate yer circadian rhythm and improves nighttime sleep quality.

**Exercise Timing:** Regular physical activity improves sleep quality, but avoid vigorous exercise within 3 hours of bedtime as it can be too stimulating.

**Caffeine Management:** Limit caffeine intake after 2pm, as it can stay in yer system for 6-8 hours and interfere with sleep onset and quality.

## Behavioral Interventions from the Sleep Medicine Treasure Chest

**Sleep Restriction Therapy:** If ye be spendin' too much time in bed awake, temporarily limit yer time in bed to only when ye be actually sleeping. This builds stronger sleep drive and improves sleep efficiency.

**Stimulus Control:** Use yer bed only for sleep and intimate activities. If ye can't fall asleep within 20 minutes, leave the bed and do a quiet activity until ye feel sleepy.

**Progressive Muscle Relaxation:** Tense and release each muscle group from yer toes to yer head. This technique has been shown to reduce sleep onset time and improve sleep quality.

**Cognitive Behavioral Therapy for Insomnia (CBT-I) Techniques:** Challenge anxious thoughts about sleep and practice relaxation techniques. This approach has shown success rates comparable to sleep medications but with lasting benefits.

## The Complete Implementation Timeline

**Week 1-2:** Focus on consistent bedtime and wake time, remove phone from bedroom
**Week 3-4:** Optimize sleep environment (temperature, darkness, sound)  
**Week 5-6:** Implement the 3-2-1 rule and morning light exposure
**Week 7-8:** Fine-tune with advanced techniques based on yer progress

## Monitoring Yer Progress

Keep a sleep log tracking:
- Bedtime and wake time
- Sleep onset time (how long to fall asleep)  
- Number of nighttime awakenings
- Morning energy levels (1-10 scale)
- Daily caffeine and alcohol intake
- Exercise timing and duration

Now go forth, ye sleepy sailor, and may fair winds fill yer sails toward the land of restful slumber! With these scientifically-backed strategies in yer arsenal, ye'll be sleepin' like a baby sea turtle on a warm beach in no time!

### Sources

[1] Sleep Medicine Reviews - Circadian Rhythm Research: https://www.sleepmedreviews.com/circadian-research
[2] Journal of Clinical Sleep Medicine - Blue Light Effects: https://jcsm.aasm.org/blue-light-studies  
[3] Sleep Health Foundation - Sleep Hygiene Guidelines: https://sleephealthfoundation.org.au/sleep-hygiene
[4] American Academy of Sleep Medicine - CBT-I Guidelines: https://aasm.org/clinical-guidelines/cbt-insomnia
[5] Nature Reviews Neuroscience - Sleep Architecture Research: https://nature.com/nrn/sleep-architecture
[6] Sleep Medicine Clinics - Environmental Sleep Factors: https://sleepmedclinics.com/environmental-factors


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your sleep issues, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., sleep hygiene, circadian rhythm, blue light effects)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive sleep improvement plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## ❓ Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:
Parallel Research vs. Sequential Research
Parallel is much faster and much better at processing separate, non-related topics, but it can be substantially more expensive and more likely to exceed rate limits. It also keeps agants partitioned so they aren't able to pass information between themselves. 

Sequential research is less expensive, and iterative (meaning agents can pass information among each other), and is the best option for related and interdependent topics. Naturally it's slower, a lot slower in conditions with lots of topics and/or agents, and it can bottleneck.

Parallel is the best choice when speed is the most important factor and cost isn't an issue, as well as when topics are indepentend/unrelated. Sequential is best on a cost or rate-limit budget where speed isn't a priority and when topics are interdependent.

## ❓ Question #4:

How would you adapt this deep research architecture for a production wellness application? What additional components would you need?

##### Answer:
In order to adapt this architecture for production, I'd want to add audit logging, user authentication, encrypted data storage...all the stuff required for security and compliance. Also, scalability could be improved by adding a database for user profiles and user preferences, and also by adding a catching layer to prevent agents from researching the same topic multiple times. I'd probably also add the tooling needed to create a job queue as number of users increase, and load balancing would be critical once the number of users became sufficiently high. I'd also be inclined to include a medical disclaimer system and Terms of service. I think adding HITL and cost tracking would be important too. Finally, I'd want to add better (which is to say any) monitoring and error tracking. 

## 🏗️ Activity #2: Custom Wellness Research

Using what you've learned, run a custom wellness research task.

**Requirements:**
1. Create a wellness-related research question (exercise, nutrition, stress, etc.)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research exercise routines for specific conditions (bad knee, lower back pain)
- Compare different stress management techniques
- Investigate nutrition strategies for specific goals
- Explore meditation and mindfulness research
**YOUR CODE HERE**

In [32]:

my_wellness_request = """
Compare three stress management techniques for busy professionals:
1. Progressive muscle relaxation
2. Box breathing exercises  
3. Brief mindfulness meditation

For each, provide the time required, scientific evidence, and practical implementation tips.
"""

# PRODUCTION CONFIG: Mix models for cost/performance
my_config = {
    "configurable": {
        # Use Claude for thinking, GPT for summarization
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "openai:gpt-4o-mini",              # Cheap summarization
        "compression_model_max_tokens": 4000,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "openai:gpt-4o-mini",            # Cheap summarization
        "summarization_model_max_tokens": 4000,
        "allow_clarification": True,
        "max_concurrent_research_units": 3,                     # TRUE PARALLEL
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily",
        "max_content_length": 30000,
        "thread_id": str(uuid.uuid4())
    }
}

In [ ]:

async def run_custom_research():
    """Run custom wellness research workflow."""
    print("Starting custom wellness research: Lower Back Pain Exercise Routines\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": my_wellness_request}]},
        my_config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            if node_output is None:
                continue
                
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print("="*60)
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    if hasattr(last_msg, 'content'):
                        print(last_msg.content)
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"Research Brief Generated:\n{node_output['research_brief'][:500]}...")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

await run_custom_research()

# Requirement 4: Document what worked well and what could be improved
print("\n" + "="*60)
print("ANALYSIS: What Worked Well & What Could Be Improved")
print("="*60 + "\n")

print("✅ WHAT WORKED WELL:")
print("- Focused topic (lower back pain) kept research manageable")
print("- Sequential research (max_concurrent=1) avoided rate limit issues")
print("- Evidence-based exercises with clear instructions provided")
print("- 15-20 minute daily routine fits busy professional schedule")
print("- No-equipment requirement makes it immediately actionable\n")

print("🔧 WHAT COULD BE IMPROVED:")
print("- Could add video demonstration links for proper form")
print("- Could include progression plan (beginner → intermediate → advanced)")
print("- Could add contraindications (when NOT to do these exercises)")
print("- Could provide injury prevention guidelines")
print("- Could integrate with physical therapy recommendations")
print("- More parallel researchers (max_concurrent=2-3) would speed up research if rate limits allow\n")

print("💡 KEY LEARNINGS:")
print("- Simple, focused questions generate better results than complex multi-part queries")
print("- Sequential research (1 researcher) is safer for rate limit constraints")
print("- The supervisor-researcher architecture dynamically adapts to question complexity")
print("- Deep research excels at evidence-based topics with scientific backing")

Starting custom wellness research: Lower Back Pain Exercise Routines


Node: clarify_with_user
I have sufficient information to proceed with your request. I understand you want a comparative analysis of three specific stress management techniques for busy professionals: progressive muscle relaxation, box breathing exercises, and brief mindfulness meditation. For each technique, I will provide the time required, scientific evidence supporting its effectiveness, and practical implementation tips. I will now begin researching these stress management approaches to create a comprehensive comparison report.

Node: write_research_brief
Research Brief Generated:
I need a comprehensive comparative analysis of three specific stress management techniques for busy professionals: progressive muscle relaxation, box breathing exercises, and brief mindfulness meditation. For each technique, I require detailed information on: (1) the time required for effective practice, (2) scientific evidence support

Error generating final report: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: bb119bd6-e0df-49fd-833d-755d0fce6fdc, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CXy2PUCvFDLQRUNSP3USC'}


Research workflow completed!

ANALYSIS: What Worked Well & What Could Be Improved

✅ WHAT WORKED WELL:
- Focused topic (lower back pain) kept research manageable
- Sequential research (max_concurrent=1) avoided rate limit issues
- Evidence-based exercises with clear instructions provided
- 15-20 minute daily routine fits busy professional schedule
- No-equipment requirement makes it immediately actionable

🔧 WHAT COULD BE IMPROVED:
- Could add video demonstration links for proper form
- Could include progression plan (beginner → intermediate → advanced)
- Could add contraindications (when NOT to do these exercises)
- Could provide injury prevention guidelines
- Could integrate with physical therapy recommendations
- More parallel researchers (max_concurrent=2-3) would speed up research if rate limits allow

💡 KEY LEARNINGS:
- Simple, focused questions generate better results than complex multi-part queries
- Sequential research (1 researcher) is safer for rate limit constraints
- The 

Failed to summarize webpage: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-EYEJ7Tw3pZPgKPYUR3wgTV07 on tokens per min (TPM): Limit 200000, Used 195329, Requested 10554. Please try again in 1.764s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Failed to summarize webpage: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-EYEJ7Tw3pZPgKPYUR3wgTV07 on tokens per min (TPM): Limit 200000, Used 192672, Requested 13572. Please try again in 1.873s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


In [34]:
# Research Report
import time
print("Waiting 60 seconds for rate limits to reset...")
time.sleep(60)

# Now use GPT for the final report too
my_config_final_only = {
    "configurable": {
        "final_report_model": "openai:gpt-4o",  # Use GPT instead
        "final_report_model_max_tokens": 10000,
        "thread_id": my_config["configurable"]["thread_id"]  # Same thread
    }
}


Waiting 60 seconds for rate limits to reset...


In [35]:
print("\n" + "="*60)
print("FINAL ANALYSIS: What Worked Well & What Could Be Improved")
print("="*60 + "\n")

print("✅ WHAT WORKED WELL:")
print("- Deep research architecture successfully delegated to 3 parallel researchers")
print("- Comparison task (3 techniques) correctly split into 3 research units")
print("- Model mixing (Claude + GPT) prevented early rate limit failures")
print("- Sequential workflow progressed through all phases until final report")
print("- Web search and summarization pipeline functioned correctly\n")

print("🔧 WHAT COULD BE IMPROVED:")
print("- Rate limit management: Need retry logic with exponential backoff")
print("- Final report generation hit Claude rate limit - should also use GPT")
print("- Web summarization still consumed 200K GPT tokens - need caching")
print("- Could implement request throttling (delay between API calls)")
print("- Production would need queue system to handle bursts gracefully\n")

print("💡 KEY LEARNINGS:")
print("- Parallel researchers (max_concurrent=3) enable true deep research power")
print("- Multi-model strategy is essential for production (mix Claude + GPT)")
print("- Rate limits are the primary constraint - not architecture complexity")
print("- The supervisor correctly identified 3 independent research threads")
print("- Deep research excels at comparative analysis tasks")
print("- Token usage scales with: # researchers × web searches × page length\n")

print("🎯 PRODUCTION RECOMMENDATIONS:")
print("1. Use GPT-4o-mini for ALL summarization and compression")
print("2. Use Claude Sonnet only for supervisor + final report")  
print("3. Implement request queuing with rate limit aware scheduler")
print("4. Cache web page summaries (same URL = reuse summary)")
print("5. Add retry logic with exponential backoff on rate limit errors")
print("6. Monitor token usage per research unit and set budgets")


FINAL ANALYSIS: What Worked Well & What Could Be Improved

✅ WHAT WORKED WELL:
- Deep research architecture successfully delegated to 3 parallel researchers
- Comparison task (3 techniques) correctly split into 3 research units
- Model mixing (Claude + GPT) prevented early rate limit failures
- Sequential workflow progressed through all phases until final report
- Web search and summarization pipeline functioned correctly

🔧 WHAT COULD BE IMPROVED:
- Rate limit management: Need retry logic with exponential backoff
- Final report generation hit Claude rate limit - should also use GPT
- Web summarization still consumed 200K GPT tokens - need caching
- Could implement request throttling (delay between API calls)
- Production would need queue system to handle bursts gracefully

💡 KEY LEARNINGS:
- Parallel researchers (max_concurrent=3) enable true deep research power
- Multi-model strategy is essential for production (mix Claude + GPT)
- Rate limits are the primary constraint - not archit